# Start Here — NVIDIA Physical AI Masterclass
## Deploy → Test → Demonstrations → Train → Retest → Compare
Use **this notebook only**, from top to bottom, in your cloned GitHub repository.
Each code cell runs with **Shift+Enter**. Keep one operator per instance.

**Fresh environment:** the same Brev Ubuntu 22.04 image, two RTX PRO 6000 Blackwell
96 GB GPUs, working NVIDIA driver, Docker + NVIDIA Container Toolkit, passwordless
sudo, internet/model access and a Python 3.12 Jupyter kernel. The infrastructure
checks below stop if the image differs; they do not provision another VM.

**Timing:** first-time container/model downloads, compilation and preparation are
additional to the **90-minute learning lab**. Budget a separate deployment session;
do not promise cold setup fits in 90 minutes. Prepared environments skip setup below.
Models, videos and credentials are not included in Git. Allow 180 GiB free in the
repository filesystem before a cold Isaac/training install, plus the base installer's
300 GiB Docker/data storage check. These are separate checks, not a total disk-size promise.

The simulator installers accept NVIDIA software terms. Review the linked model and
asset licenses in ISAAC_DEPLOYMENT.md before running them. No physical robot is controlled.

### Setup A — Locate the repository and choose fresh or prepared mode
**Purpose:** use the notebook's Python environment and the existing deployment scripts.
**Expected:** repository path and a setup flag. `RUN_SETUP` defaults to True; set it
to False **only for a prepared instance** to skip installers and continue to readiness.
Setup can restart services. Finish or release any active experiment first.

**What the next code cell does**
- Confirms that Jupyter is using Python 3.12, then finds the repository root by looking for the Isaac installer.
- Adds this repository and the active Python environment to the import and command search paths.
- Defines `run()`, `script()` and `assert_idle()` helpers used by every later setup cell.
- Sets `RUN_SETUP=True` for a fresh VM. The idle check prevents deployment changes while a training or simulator job is active.

**Keywords:** a **kernel** is the Python process executing notebook cells; `PATH` selects command-line programs; `sys.path` selects Python modules; an **API endpoint** is a URL exposed by a service; `localhost` or `127.0.0.1` means the service is reachable only on this VM.

In [ ]:
from pathlib import Path
import json, os, sys, subprocess, time, urllib.request
assert sys.version_info[:2] == (3, 12), 'Select the Python 3.12 Jupyter kernel.'
candidates = [Path.cwd(), *Path.cwd().parents, Path.home()/'vss-robotics-workshop',
              Path.home()/'EMEA-Masterclass-Physical-AI-Workshop']
ROOT = next((p for p in candidates if (p/'workshop/scripts/install_isaac.py').is_file()), None)
assert ROOT is not None, 'Open this notebook inside the cloned repository, then restart the kernel.'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ['PATH'] = str(Path(sys.executable).parent) + os.pathsep + os.environ['PATH']
RUN_SETUP = True  # False only when this instance has already been fully prepared.
def run(*args):
    subprocess.run([str(a) for a in args], cwd=ROOT, check=True)
def script(name, *args):
    run(sys.executable, ROOT/'workshop/scripts'/name, *args)
def assert_idle():
    for route in ('state', 'training/state'):
        try:
            with urllib.request.urlopen('http://127.0.0.1:8097/mission-api/'+route, timeout=3) as response:
                state = json.load(response)
        except OSError:
            continue  # The API is absent on a fresh machine.
        assert not state.get('busy') and not state.get('restore_nemotron'), 'Finish or release the active experiment first.'
print('Repository:', ROOT)
print('Run deployment stages:', RUN_SETUP)

### Setup B — Check the host and install build tools
**NVIDIA component:** GPU driver/container runtime for the VSS model services and Isaac.
This verifies the existing Brev image and installs compiler/media tools plus pinned uv.
The base installer checks GPU type, driver, Docker compatibility and storage.
**Expected:** preflight passed. If a prerequisite fails, resolve it on this VM before continuing.

**What the next code cell does**
- Verifies passwordless `sudo`, both GPUs, Docker, and NVIDIA Container Toolkit before downloading anything large.
- Checks free storage, installs Linux build and video tools, bootstraps `pip` when the Brev kernel does not contain it, and pins `uv==0.12.13`.
- Runs the project preflight, which validates the expected RTX PRO 6000 GPUs, driver, Docker version, storage and Compose graph.

**Keywords:** the **GPU driver** lets Linux communicate with the GPU; **NVIDIA Container Toolkit** gives containers controlled GPU access; **Docker Compose** describes a set of connected containers; `pip` installs Python packages; **uv** reproducibly creates Python environments from locked dependencies; **GiB** is binary storage capacity.

In [ ]:
if RUN_SETUP:
    import shutil
    assert_idle()
    assert sys.platform == 'linux', 'Run on the Brev Linux instance.'
    run('sudo', '-n', 'true')
    run('nvidia-smi')
    run('docker', 'info', '--format', '{{.ServerVersion}}')
    run('nvidia-container-cli', '--version')
    arena_python = ROOT/'third_party/IsaacLab-Arena/.venv/bin/python'
    required_gib = 45 if arena_python.exists() else 180
    free_gib = shutil.disk_usage(ROOT).free / 1024**3
    assert free_gib >= required_gib, f'Repository has {free_gib:.1f} GiB free; setup needs {required_gib} GiB.'
    run('sudo', 'apt-get', 'update', '-qq')
    run('sudo', 'apt-get', 'install', '-y', 'build-essential', 'git', 'git-lfs',
        'cmake', 'ninja-build', 'pkg-config', 'ffmpeg', 'curl')
    pip_check = subprocess.run([sys.executable, '-m', 'pip', '--version'],
        cwd=ROOT, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if pip_check.returncode:
        print('This Jupyter environment has no pip; bootstrapping it with Python ensurepip.')
        run(sys.executable, '-m', 'ensurepip', '--upgrade')
    run(sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', 'uv==0.12.13')
    script('deploy_lab.py', 'check')
else:
    print('Prepared mode: build tools and installers skipped.')

### Setup C — Enter private NVIDIA and Hugging Face credentials
**Purpose:** NVIDIA/NGC registry access pulls the NIM and VSS images; HF access downloads
the pinned GR00T foundation, task checkpoint and demonstrations. Use a read token with
access to these resources. Accept any required terms on their model pages first.
**Expected:** HF access verified and credentials saved outside Git with private permissions.
Prompts are hidden. Never paste keys into cell source, chat or screenshots. NVIDIA registry
entitlements are checked by the next deployment cell. Prepared mode skips these prompts;
set `UPDATE_CREDENTIALS=True` if this team needs replacement credentials.

**What the next code cell does**
- Collects the NGC key and Hugging Face read token through hidden prompts and writes them outside Git with owner-only permissions.
- Calls the pinned foundation model, task checkpoint and dataset endpoints to prove that the token has the required access.
- Keeps `UPDATE_CREDENTIALS=False` during normal reruns so saved credentials are not overwritten accidentally.

**Keywords:** **NGC** is NVIDIA's catalog and registry for containers and models; a **registry key** authenticates Docker image pulls; a **Hugging Face read token** authorizes model and dataset downloads; a **revision** pins an exact immutable model version; file modes `0700` and `0600` restrict access to the VM owner.

In [ ]:
UPDATE_CREDENTIALS = False
if RUN_SETUP or UPDATE_CREDENTIALS:
    import getpass, urllib.error
    private = Path.home()/'.local/state/vss-robotics-workshop'
    private.mkdir(parents=True, exist_ok=True, mode=0o700)
    private.chmod(0o700)
    def save_private(name, value):
        fd = os.open(private/name, os.O_WRONLY|os.O_CREAT|os.O_TRUNC, 0o600)
        with os.fdopen(fd, 'w') as stream:
            os.fchmod(stream.fileno(), 0o600)
            stream.write(value)
    ngc = getpass.getpass('NVIDIA/NGC registry key (hidden): ').strip()
    assert ngc and '\n' not in ngc and '\r' not in ngc, 'Enter a nonempty single-line key.'
    save_private('ngc.key', ngc)
    del ngc
    hf = getpass.getpass('Hugging Face read token (hidden): ').strip()
    try:
        profile = json.loads((ROOT/'workshop/training_profile.json').read_text())
        urls = [
            'https://huggingface.co/nvidia/GR00T-N1.6-3B/resolve/'+profile['foundation_revision']+'/config.json',
            'https://huggingface.co/api/models/nvidia/GN1.6-Tuned-Arena-GR1-PlaceItemCloseDoor-Task/auth-check',
            'https://huggingface.co/api/datasets/nvidia/Arena-GR1-Manipulation-PlaceItemCloseDoor-Task/auth-check']
        for url in urls:
            req = urllib.request.Request(url, headers={'Authorization': 'Bearer '+hf})
            try:
                with urllib.request.urlopen(req, timeout=30) as response: response.read()
            except urllib.error.HTTPError as exc:
                raise RuntimeError(f'HF access denied (HTTP {exc.code}); check token scope and model terms.') from None
        save_private('hf.token', hf)
        print('HF access verified. Credentials saved privately.')
    finally:
        del hf
        if 'req' in globals(): del req

### Setup D — Deploy VSS, Cosmos and Nemotron
The base installer pulls pinned containers, configures this instance’s storage and URLs, and waits for model services. First start can take tens of minutes. Expected: healthy services; use the private deployment log on the VM for errors.
Run this cell once the preceding stage succeeds. In prepared mode it is skipped.

**What the next code cell does**
- Runs the idempotent base deployment: it pulls pinned images, creates private environment files, starts containers and waits for health checks.
- Starts Cosmos as the visual reasoner, Nemotron as the language model used by the VSS agent, VSS services for video evidence, and the supporting message/data services.
- Publishes only the workshop gateway on port 7777; model and infrastructure ports remain on localhost.

**Keywords:** a **NIM** is an NVIDIA Inference Microservice that packages an optimized model server; **VSS** is the Video Search and Summarization blueprint; **Cosmos** reasons about physical-world video; **Nemotron** provides language reasoning for the VSS agent; **Kafka** is an event stream that passes video and metadata messages between services; **Redis** is a fast in-memory coordination store; **VST/VIOS** ingests and catalogs video; **HAProxy** is the port-7777 gateway; an **idempotent** deployment can be rerun without intentionally duplicating the stack.

In [ ]:
if RUN_SETUP:
    assert_idle()
    script('deploy_lab.py', 'deploy')
else:
    print('Prepared mode: stage skipped.')

### Setup E — Install Isaac and the matching GR00T policy
Create isolated simulator/policy environments, download the pinned GR1 checkpoint and verify robot/scene assets. CUDA 12.8 supports this Blackwell GPU. Compilation and downloads may take a long time. Expected: Isaac profile prepared.
Run this cell once the preceding stage succeeds. In prepared mode it is skipped.

**What the next code cell does**
- Clones exact Isaac Lab Arena, Isaac Lab and Isaac-GR00T revisions and creates separate simulator and policy Python environments.
- Installs CUDA 12.8 PyTorch support for Blackwell, downloads the published GR1 task policy and verifies every robot, bottle and kitchen asset against a SHA-256 manifest.
- Keeps simulator dependencies separate from GR00T dependencies so their Python and CUDA requirements cannot overwrite each other.

**Keywords:** **Isaac Sim** is NVIDIA's GPU-accelerated robotics simulator; **Isaac Lab** supplies robot-learning environments; **Arena** provides this task and its evaluation workflow; **GR00T** is NVIDIA's robot foundation-model family; a **policy** maps observations to actions; a **checkpoint** is a saved set of model weights; **CUDA** runs computation on NVIDIA GPUs; a **SHA-256 manifest** detects changed assets byte for byte.

In [ ]:
if RUN_SETUP:
    assert_idle()
    script('install_isaac.py')
else:
    print('Prepared mode: stage skipped.')

### Setup F — Prove the official task executes
Run three real episodes with NVIDIA’s original task checkpoint. Review results in workshop/results/isaac-baseline. At least one successful task is required by the next activation gate; an execution failure is not replaced with a recording.
Run this cell once the preceding stage succeeds. In prepared mode it is skipped.

**What the next code cell does**
- Starts the original GR00T checkpoint as a local inference server and launches three fresh headless Isaac episodes.
- Sends camera images and robot state to GR00T, applies returned action chunks in physics, records video, and measures task completion.
- Uses the headless Matplotlib backend so Jupyter's inline display setting cannot break Isaac startup. Activation is allowed only after at least one measured success.

**Keywords:** an **episode** is one task attempt from reset to termination; **inference** runs fixed weights to predict actions and does not train; **closed loop** means the policy repeatedly observes the changed scene and predicts again; an **action chunk** is a short sequence predicted in one inference call; a **control step** applies one action to the simulator; a **predicate** is a simulator truth test such as bottle-in-fridge or door-closed; **headless** rendering creates images without a desktop window.

In [ ]:
if RUN_SETUP:
    assert_idle()
    script('validate_isaac_baseline.py')
else:
    print('Prepared mode: stage skipped.')

### Setup G — Start the shared mission services
Install the persistent GR00T policy and notebook/browser API after baseline validation. This restarts only the relevant workshop services and refreshes the UI. Expected: services active on this instance.
Run this cell once the preceding stage succeeds. In prepared mode it is skipped.

**What the next code cell does**
- Installs persistent local services for the GR00T policy and the shared mission API after the baseline gate passes.
- Connects the notebook and browser UI to the same experiment state, while keeping the policy and API ports private to the VM.
- Refreshes the UI container without rebuilding the validated model and simulator environments.

**Keywords:** **systemd** keeps a Linux service running across terminal and notebook sessions; an **API** is the programmatic interface used by both clients; **state** records the current experiment phase and artifacts; **CSRF token** binds browser actions to the current service session; a **port** identifies a network service, with 8097 used locally by this mission API.

In [ ]:
if RUN_SETUP:
    assert_idle()
    script('activate_isaac.py')
else:
    print('Prepared mode: stage skipped.')

### Setup H — Prepare demonstrations and the early checkpoint
Download the foundation weights and 100 demonstrations, then perform 200 real optimizer steps to create the early policy. A complete early checkpoint is retained on rerun. An incomplete one requires inspection. Expected: prepared training inputs; attendees later perform 2,000 additional steps.
Run this cell once the preceding stage succeeds. In prepared mode it is skipped.

**What the next code cell does**
- Downloads the pinned GR00T foundation weights and 100 task demonstrations, then verifies their recorded revision.
- Performs 200 supervised optimizer steps to produce the deliberately early checkpoint used for the before-training test.
- Reuses a complete prepared checkpoint on rerun and refuses to present incomplete training output as valid.

**Keywords:** a **foundation model** has broad pretrained capability; a **demonstration** is a time-aligned example of images, state, instruction and correct actions; **supervised fine-tuning** adjusts weights toward demonstration actions; an **optimizer step** computes gradients from one batch and updates weights once; a **batch** groups multiple samples per update; an **early checkpoint** is intentionally only partly adapted, not an untrained robot.

In [ ]:
if RUN_SETUP:
    assert_idle()
    script('prepare_training_lab.py')
else:
    print('Prepared mode: stage skipped.')

### Setup I — Verify the deployment before learning
**Purpose:** verify package tests, the real API and the early checkpoint. The baseline above
is a real inference check; tests alone cannot establish GPU readiness.
**Expected:** checks pass and the training API responds. Now continue directly through the
90-minute learning cells below. You can use port 7777 `/learn.html` for the same experiment.
After the final comparison, the last cell activates the root UI for subsequent attendees.

**What the next code cell does**
- Parses every notebook code cell, runs the 30 repository tests, checks the early checkpoint and waits up to five minutes for the mission API.
- Allows notebook outputs during this live run while retaining source and security checks for the release package.
- Prints the API phase and the browser location only after the deployment is ready for the learning experiment.

**Keywords:** a **unit test** checks a bounded software behavior; a **readiness check** proves a service can answer requests; a **timeout** prevents an infinite wait; a **service phase** is the API's explicit lifecycle state; package validation proves software integrity but does not replace real GPU inference.

In [ ]:
script('verify_package.py', '--allow-notebook-outputs')  # Jupyter may autosave this running notebook.
run(sys.executable, '-m', 'unittest', 'discover', '-s', 'workshop/tests')
profile = json.loads((ROOT/'workshop/training_profile.json').read_text())
assert (ROOT/profile['before_checkpoint']/'config.json').is_file(), 'Run Setup H first.'
deadline = time.monotonic()+300
while True:
    try:
        with urllib.request.urlopen('http://127.0.0.1:8097/mission-api/training/state', timeout=5) as response:
            state = json.load(response)
        break
    except OSError:
        if time.monotonic() > deadline: raise RuntimeError('Training API is not ready. Run Setup G after a successful Setup F, then inspect physical-ai-isaac service logs if activation fails.') from None
        time.sleep(5)
print('Training API phase:', state['phase'])
print('Ready for the learning experiment below. Open the actual Brev Secure Link for port 7777 /learn.html.')

# NVIDIA Physical AI Masterclass
## Test. Demonstrate. Train. Act.
**90 minutes on a prepared Brev clone + optional experiments**

Teach a GR1 robot to place a bottle in a refrigerator and close the door. You will test an early GR00T checkpoint,
inspect demonstrations, perform real supervised fine-tuning, and test the newly saved weights in Isaac Sim.
The early checkpoint is already foundation-pretrained and has received 200 task optimizer steps. It is not a blank robot.
We never inject a fake failure or promise that a short training run succeeds on every scene.

The browser on **Brev Secure Link port 7777** and this notebook share one experiment. Use one operator per instance.
Deployment is covered above in this notebook. No physical robot commands are sent.

## 1. Understand the learning loop — 10 minutes
**Purpose:** distinguish learning from inference.

NVIDIA's three-computer architecture separates training, simulation and deployed robot inference. This workshop
co-locates training and simulation on Brev GPUs; the physical robot computer is a future integration.

**GR00T N1.6** predicts robot actions from images, joint states and language. Its weights change during fine-tuning.
**Isaac Sim + Isaac Lab Arena** execute those actions through physics and measure task predicates.
**Cosmos** interprets the resulting video; it does not update the robot policy. **VSS** indexes episode evidence.
**Nemotron** supports the VSS agent used to register episode evidence; it is not the GR00T training optimizer.

For developers, the reusable outcome is a task-adapted policy plus evidence for deciding whether it is ready for broader evaluation.
Simulation success alone does not establish hardware readiness or production licensing.

**What the next code cell does**
- Locates the prepared repository, imports the small `TrainingMission` client and connects it to the localhost mission API.
- Creates no simulator and changes no weights; it only gives Python methods such as `create()`, `before()`, `train()` and `after()` to the notebook.

**Keywords:** a **client** sends requests to a service; `TrainingMission` is the typed notebook wrapper around HTTP calls; the **three-computer architecture** separates training, simulation and deployed robot execution even though this lab co-locates the first two; **model weights** are learned numeric parameters.

In [ ]:
from pathlib import Path
import json, sys, time
from IPython.display import display, Video, clear_output
ROOT=next(p for p in [Path.cwd(),Path.cwd().parent,Path.home()/'vss-robotics-workshop']
          if (p/'workshop/lab/training.py').exists())
sys.path.insert(0,str(ROOT))
from workshop.lab.training import TrainingMission
lab=TrainingMission()
print('Notebook connected to the same experiment API as the browser.')

## 2. Verify the prepared deployment — 10 minutes
Setup A–I above prepares this instance. Confirm the model files and mission API below before starting the experiment.
The optional hidden HF prompt is only needed to replace model access credentials. NVIDIA credentials were configured in Setup C.

**What the next code cell does**
- Optionally replaces the private Hugging Face token and verifies access to the exact pinned GR00T revision.
- Loads `training_profile.json`, confirms that the early checkpoint contains its model configuration, and prints the mission-service phase.
- Does not download or train a model when `ENTER_HF_TOKEN=False`.

**Keywords:** a **profile** is the versioned configuration joining model revisions, dataset paths and training settings; `config.json` describes checkpoint architecture and loading parameters; a **service phase** such as `idle`, `ready`, `training` or `complete` makes progress explicit.

In [ ]:
ENTER_HF_TOKEN=False
if ENTER_HF_TOKEN:
    import getpass, os, urllib.request
    token=getpass.getpass('Hugging Face read token (hidden): ').strip()
    req=urllib.request.Request('https://huggingface.co/nvidia/GR00T-N1.6-3B/resolve/d0814e7ecb19202e7c8468b46098b0b7ef3a6d61/config.json',
                              headers={'Authorization':'Bearer '+token})
    with urllib.request.urlopen(req,timeout=30) as response: response.read()
    private=Path.home()/'.local/state/vss-robotics-workshop'
    private.mkdir(parents=True,exist_ok=True,mode=0o700)
    fd=os.open(private/'hf.token',os.O_WRONLY|os.O_CREAT|os.O_TRUNC,0o600)
    with os.fdopen(fd,'w') as stream: stream.write(token)
    del token
    print('HF model access verified; saved privately.')
profile=json.loads((ROOT/'workshop/training_profile.json').read_text())
assert (ROOT/profile['before_checkpoint']/'config.json').exists(), 'Instructor preparation is incomplete'
print(json.dumps({'profile':profile,'service_phase':lab.status()['phase']},indent=2))

## 3. Test the early checkpoint — 15 minutes
**Purpose:** measure capability before this session's training.

Set `SEED` to an integer for a repeatable scene, or leave it as `None` to record a fresh one.
The same scene will be used after training. The bottle position varies within the task's bounded workspace;
new seeds do not guarantee dramatically different trajectories.

**Expected output:** a fresh video, action telemetry and a success/failure predicate. A control step applies one action
at 50 Hz simulation time. GR00T supplies chunks of 16 actions before another inference call.
These control steps do not change model weights.

**What the first code cell does**
- Creates an experiment and records every randomized scene parameter. `SEED=None` samples and stores a new seed; an integer reproduces the same initial scene.
- Runs the early checkpoint in a fresh Isaac episode, waits for completion, prints measured telemetry and displays the video generated on this VM.

**What the second code cell does**
- Opens the first recorded policy interaction and shows the language instruction, image shape, joint-state dimensions, predicted action chunk and first executed action.
- Prints the exact checkpoint used so behavior can be traced to a model artifact.

**Keywords:** a **seed** initializes pseudorandom scene generation; an **observation** combines camera images and robot state; **joint positions** describe the robot's configuration; an **action dimension** is one commanded control value; an **inference call** is one forward pass through GR00T; an **action hash** fingerprints the executed sequence so before/after behavior can be compared.

In [ ]:
SEED=None
experiment=lab.create(seed=SEED)
print('Recorded seed:',experiment['scenario']['seed'])
lab.before()
before=lab.wait()['before']
print({k:before[k] for k in ['success','episode_steps','inference_calls','action_sha256']})
folder=ROOT/'workshop/results/training'/experiment['session_id']/'before'/before['run_id']
display(Video(str(folder/'episode.mp4'),embed=True))

In [ ]:
# Inspect the simulator observation and the adapter's returned action chunk.
sample=before['policy_io'][0]
print('Instruction:',sample['instruction'])
print('Camera shape:',sample['camera_shape'])
print('Raw simulator state dimensions:',len(sample['joint_positions'][0]))
print('Simulator action dimensions:',len(sample['executed_action'][0]))
print('Actions in this chunk:',len(sample['predicted_chunk'][0]))
print('Joint state:',sample['joint_positions'])
print('First executed action:',sample['executed_action'])
print('Checkpoint:',before['versions']['checkpoint'])

## 4. Inspect successful demonstrations — 10 minutes
**Purpose:** understand the supervised data that teaches the task.

The pinned NVIDIA Arena dataset contains 100 GR1 trajectories: ego-view video, joint states, action targets and an instruction.
Its state/action convention has 26 values. The Arena adapter maps the simulator's fuller joint state into that convention
and maps policy predictions back to simulator actions. The 54 raw state values and 36 applied action values in the previous
cell describe simulator interfaces, not the training batch size or the 16-action chunk length.
All 100 demonstrations are training inputs in this exercise. The simulator tests are fresh closed-loop episodes;
we do not claim a held-out demonstration loss or out-of-distribution generalization.

**Expected output:** dataset metadata and a clearly labeled demonstration video. This is the only prerecorded training
example in the flow; baseline and retest videos are generated by your own simulator runs.

**What the next code cell does**
- Reads the LeRobot dataset metadata, prints robot type, episode/frame counts and frame rate, then displays demonstration episode 0.
- Labels the video as prerecorded training input so it cannot be confused with live policy evaluation.

**Keywords:** a **trajectory** is the ordered observation/action history of an episode; **ego view** is the robot-mounted camera; **FPS** is frames per second; **LeRobot** is the dataset layout used for time-aligned robotics data; an **action target** is the demonstrated control the model learns to predict; **training data** teaches weights, while **evaluation episodes** measure the resulting behavior.

In [ ]:
DATA=ROOT/'third_party/training-inputs/dataset/ranch_bottle_into_fridge/ranch_bottle_into_fridge_generated_100/lerobot'
info=json.loads((DATA/'meta/info.json').read_text())
print({k:info[k] for k in ['robot_type','total_episodes','total_frames','fps']})
display(Video(str(DATA/'videos/chunk-000/observation.images.ego_view/episode_000000.mp4'),embed=True))
print('Training demonstration 0 — not a live evaluation.')

## 5. Train GR00T — 25 minutes
**Purpose:** produce new policy weights using real gradient updates.

Choose the number of optimizer steps below. A step processes a batch of 16 demonstration samples.
The live run continues from the early checkpoint with a fresh optimizer. It updates the top four language layers,
projector, diffusion action head and vision-language normalization; the visual encoder stays frozen.

**Expected output:** measured action loss, elapsed time, checkpoint path and changed weight hashes.
Loss reduction does not prove task success. The experiment keeps GPU 0 reserved across baseline, training and retest;
Nemotron is restored after comparison, stop, or an explicit GPU release. Leave this cell running or inspect the same progress in the browser.

**What the next code cell does**
- Requests 2,000 optimizer steps, polls progress every five seconds and shows elapsed time plus the latest action-prediction loss.
- Continues from the 200-step early checkpoint, saves a new checkpoint and verifies that a representative weight hash changed.
- Uses a one-hour notebook timeout without silently declaring success; `lab.stop()` remains an explicit operator action.

**Keywords:** a **learning step** or optimizer step is one weight update, not one robot control step; a **gradient** indicates how each trainable weight should change to reduce error; **loss** numerically compares predicted and demonstrated actions; a **learning rate** scales each update; **frozen layers** do not change; the **diffusion action head** generates continuous action sequences; a **checkpoint path** identifies the newly saved model state; falling training loss alone does not prove task success.

In [ ]:
OPTIMIZER_STEPS=2000
lab.train(steps=OPTIMIZER_STEPS)
deadline=time.monotonic()+3600
while True:
    current=lab.status()
    progress=current.get('training') or {}
    clear_output(wait=True)
    print('Optimizer steps:',progress.get('training_steps',0),'/',OPTIMIZER_STEPS)
    print('Elapsed seconds:',progress.get('elapsed_s'))
    print('Latest action loss:',(progress.get('loss_history') or [{}])[-1].get('loss'))
    if not current['busy']: break
    if time.monotonic()>deadline: raise TimeoutError('Training still active; inspect the UI or call lab.stop()')
    time.sleep(5)
trained=lab.wait()
assert trained['training']['weights_changed'], 'Weight change was not verified'
print('New checkpoint:',trained['after_checkpoint'])
print('Before:',trained['training']['weight_before'])
print('After:',trained['training']['weight_after'])

## 6. Retest and explain the developer value — 20 minutes
**Purpose:** check whether new weights improve robot behavior under equal conditions.

The next cell starts a fresh simulator episode with the recorded scene and the same 500-step budget, loads your checkpoint,
and measures bottle placement plus door closure. Cosmos describes visible behavior; VSS indexes the video.
Model interpretation is recorded separately from the simulator result. No supervisory intervention changes this comparison.

If a service issue blocks evaluation, restore that service and rerun the evaluation cell. The failed attempt is archived;
your completed training checkpoint is retained. Do not rerun the training cell to recover an evaluation-only failure.

**Expected output:** both measured outcomes, equal-budget checks, actual action hashes and video. Improvement is an
experimental result, not a promise. Repeated failures are useful evidence about data coverage and training budget.

**What the first code cell does**
- Loads the newly trained checkpoint into a new policy server and reruns the recorded seed with the same initial bottle pose and 500-control-step budget.
- Compares measured success, latency, weight change, initial-state equality and execution budgets; then displays the newly rendered episode.
- Prints Cosmos's visual assessment and any VSS evidence-registration error separately from simulator truth.

**What the second code cell does**
- Removes the transient CSRF value, writes a versioned JSON attendee report and prints before/after action hashes.
- Preserves failures and disagreements as evidence instead of replacing them with a successful prerecorded run.

**Keywords:** an **equal-condition comparison** keeps seed, scene and maximum control budget fixed while changing only the checkpoint; a **simulator predicate** is authoritative task ground truth; **Cosmos assessment** is model interpretation of visible video; **VSS indexing** registers video and timestamps for later review; **latency** is elapsed execution time; **model disagreement** occurs when visual interpretation and simulator predicates differ; **JSON** is a machine-readable report format.

In [ ]:
lab.after()
report=lab.wait()
after=report['after']
print(json.dumps(report['comparison'],indent=2))
folder=ROOT/'workshop/results/training'/report['session_id']/'after'/after['run_id']
display(Video(str(folder/'episode.mp4'),embed=True))
print('Cosmos:',after.get('cosmos'))
print('Evidence service issues:',after.get('evidence_error'))

In [ ]:
export=ROOT/'workshop/results/training'/report['session_id']/'attendee-report.json'
report.pop('csrf',None)
export.write_text(json.dumps(report,indent=2))
print('Evidence exported:',export)
print('Before action hash:',before['action_sha256'])
print('After action hash:',after['action_sha256'])

## Optional developer exercises
1. Evaluate the same learned checkpoint across predeclared seeds with `workshop/scripts/validate_training_sweep.py`.
   Keep weights fixed when measuring scene generalization. Report every outcome and latency; do not select only wins.
2. Compare 500, 1000 and 2000 optimizer steps from the same early checkpoint. Separate learning curves from control-step counts.
3. Inspect the Cosmos prompt in `workshop/lab/training.py`. Contrast visual uncertainty with simulator ground truth.
4. Compare direct Cosmos assessment with the indexed VSS episode record and simulator predicates.
5. Inspect seeds 606 and 1010, remaining failures of the reference checkpoint in the recorded sweep. Evaluate your own checkpoint
   on that scene and discuss what extra demonstrations or training changes you would test next.

**GTM through developer outcomes:** foundation robot intelligence reduces the amount of task-specific work; fine-tuning adapts
it to demonstrations; simulation measures closed-loop behavior; VSS and Cosmos make failures reviewable. These are reusable
building blocks for evaluation and integration, rather than evidence that a production robot is ready.

**Boundaries:** no training from scratch, no guaranteed task success, no physical hardware commands, no Wandelbots integration.
The original tuned task artifact has research/evaluation restrictions. Review the foundation model, dataset and robot asset
licenses separately before production use; this workshop does not grant production rights.

Official references: [Arena policy training](https://isaac-sim.github.io/IsaacLab-Arena/main/pages/example_workflows/sequential_static_manipulation/step_4_policy_training.html),
[GR00T N1.6](https://huggingface.co/nvidia/GR00T-N1.6-3B),
[Arena dataset](https://huggingface.co/datasets/nvidia/Arena-GR1-Manipulation-PlaceItemCloseDoor-Task).

### Finish — Publish the validated training entry point
After the complete before/train/after comparison, this cell makes the learning UI the
port 7777 home page. It verifies changed weights and equal-condition evaluation first.
Activation clears the current UI experiment **but retains all reports and checkpoints**.
Run it once for a newly prepared environment; leave False when reviewing your current experiment.
Start Here remains this complete deployment notebook.

**What the next code cell does**
- Runs a release gate that requires changed weights plus a completed equal-condition before/after comparison.
- Makes the guided learning UI the port-7777 home page and creates an empty UI experiment for the next attendee.
- Retains checkpoints, videos and reports; it changes only the attendee entry point and current browser state.

**Keywords:** an **activation gate** blocks publication until required evidence exists; an **entry point** is the first page opened through Brev Secure Link; an **artifact** is a saved checkpoint, video, log or report; `ACTIVATE_ENTRY_POINT` prevents accidental UI reset while an instructor is reviewing the current experiment.

In [ ]:
ACTIVATE_ENTRY_POINT = RUN_SETUP
if ACTIVATE_ENTRY_POINT:
    script('activate_training_lab.py')
else:
    print('Existing entry point and displayed experiment retained.')
print('Use this instance’s Brev Secure Link for port 7777. Do not reuse another instance’s URL.')